In [1]:
!python --version

Python 3.13.2


In [2]:
#importing necessary modules 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#from sklearn.decomposition import PCA
#from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import seaborn as sns
import copairs
import pycytominer
from copairs.map import average_precision
from copairs.map import mean_average_precision
from utils import * 

%load_ext autoreload
%autoreload 2

/opt/anaconda3/envs/copairs/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
pip show copairs

Name: copairs
Version: 0.5.1
Summary: Find pairs and compute metrics between them
Home-page: https://github.com/cytomining/copairs
Author: 
Author-email: John Arevalo <johnarevalo@gmail.com>, Alexandr Kalinin <akalinin@broadinstitute.org>, "Alan F. Munoz" <amunozgo@broadinstitute.org>
License: 
Location: /opt/anaconda3/envs/copairs/lib/python3.13/site-packages
Requires: duckdb, pandas, statsmodels, tqdm
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [4]:
#Reading the normalized and feature selected files for copairs - fixed cells and 48h time point
profiles = {'/Users/sugan/Documents/GitHub/2022_09_07_New_phenotypic_dye_testing_CDoT_Broad_Analysis/copairs_csv/UpdatedCopairsVersion/BR00122250_normalized_feature_select_negcon_batch.csv':'Standard CP',
                                 '/Users/sugan/Documents/GitHub/2022_09_07_New_phenotypic_dye_testing_CDoT_Broad_Analysis/copairs_csv/UpdatedCopairsVersion/BR00122246_normalized_feature_select_negcon_batch.csv':'CP + MitoBrilliant',
                                 '/Users/sugan/Documents/GitHub/2022_09_07_New_phenotypic_dye_testing_CDoT_Broad_Analysis/copairs_csv/UpdatedCopairsVersion/BR00122247_normalized_feature_select_negcon_batch.csv':'CP + Phenovue phalloidin 400LS',
                                 '/Users/sugan/Documents/GitHub/2022_09_07_New_phenotypic_dye_testing_CDoT_Broad_Analysis/copairs_csv/UpdatedCopairsVersion/BR00122248_normalized_feature_select_negcon_batch.csv':'Standard CP (exposed to ChromaLive)',
                                 '/Users/sugan/Documents/GitHub/2022_09_07_New_phenotypic_dye_testing_CDoT_Broad_Analysis/copairs_csv/UpdatedCopairsVersion/BR00122249_normalized_feature_select_negcon_batch_wo_phasefeatures.csv':'ChromaLive + Hoechst'

}


In [9]:
def cell_counts(input_dict={}):
     
    combined_df = pd.DataFrame()

    for i in input_dict:
            
            with open(i, 'rb') as filetype:
                if filetype.read(2) == b'\x1f\x8b':
                    df = pd.read_csv(i, compression='gzip')
                else:
                    df = pd.read_csv(i)
            subset_df = df[['Metadata_Count_Cells', 'Metadata_Plate']]
            combined_df = pd.concat([combined_df, subset_df], axis=1)
    combined_df = pd.concat([combined_df, df['Metadata_Well']], axis=1)

    return combined_df


In [10]:
cell_counts(profiles)

,Metadata_Count_Cells,Metadata_Plate,Metadata_Count_Cells,Metadata_Plate,Metadata_Count_Cells,Metadata_Plate,Metadata_Count_Cells,Metadata_Plate,Metadata_Count_Cells,Metadata_Plate,Metadata_Well
0,3121,BR00122250,3456.0,BR00122246,3361,BR00122247,3093,BR00122248,3219,BR00122249,A01
1,3804,BR00122250,4102.0,BR00122246,4012,BR00122247,3539,BR00122248,3824,BR00122249,A02
2,3766,BR00122250,3903.0,BR00122246,4048,BR00122247,3567,BR00122248,3838,BR00122249,A03
3,2922,BR00122250,3304.0,BR00122246,3239,BR00122247,2711,BR00122248,2768,BR00122249,A04
4,3971,BR00122250,4045.0,BR00122246,3955,BR00122247,3502,BR00122248,3722,BR00122249,A05
...,...,...,...,...,...,...,...,...,...,...,...
379,4017,BR00122250,3614.0,BR00122246,3836,BR00122247,3428,BR00122248,3445,BR00122249,P20
380,3634,BR00122250,3995.0,BR00122246,3505,BR00122247,3094,BR00122248,3593,BR00122249,P21
381,3835,BR00122250,4099.0,BR00122246,3974,BR00122247,3041,BR00122248,3752,BR00122249,P22
382,4043,BR00122250,3411.0,BR00122246,4048,BR00122247,3336,BR00122248,3628,BR00122249,P23
